# **DATA LOADER AND WINE DATA CLASSIFICATION**

# **IMPORTS**

In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset,DataLoader
import numpy as np
import math

# **CUSTOM DATASET CLASS**

In [ ]:
class WineDataset(Dataset):
  def __init__(self):
    xy=np.loadtxt('/content/wine.csv',delimiter=',',skiprows=1,dtype=np.float32)
    self.x=torch.from_numpy(xy[:,1:])
    self.y = torch.from_numpy(xy[:, 0]).long() - 1
    self.n_samples=xy.shape[0]
  def __getitem__(self,index):
    return self.x[index],self.y[index]
  def __len__(self):
    return self.n_samples

In [ ]:
dataset=WineDataset()
first_data=dataset[0]
features,label=first_data
features,label

(tensor([1.4230e+01, 1.7100e+00, 2.4300e+00, 1.5600e+01, 1.2700e+02, 2.8000e+00,
         3.0600e+00, 2.8000e-01, 2.2900e+00, 5.6400e+00, 1.0400e+00, 3.9200e+00,
         1.0650e+03]),
 tensor(0))

# **MODEL CREATION**

In [ ]:
import torch.nn as nn

In [ ]:
class DataNewModel(nn.Module):
  def __init__(self,input_dim,hidden_dim,output_dim):
    super().__init__()
    self.linear1=nn.Linear(in_features=input_dim,out_features=hidden_dim)
    self.relu=nn.ReLU()
    self.linear2=nn.Linear(in_features=hidden_dim,out_features=output_dim)
  def forward(self,x):
    x=self.linear1(x)
    x=self.relu(x)
    x=self.linear2(x)
    return x



In [ ]:
model=DataNewModel(input_dim=13,hidden_dim=24,output_dim=3)
model

DataNewModel(
  (linear1): Linear(in_features=13, out_features=24, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=24, out_features=3, bias=True)
)

In [ ]:
import torch.optim as optim

In [ ]:
optimizer=optim.Adam(model.parameters(),lr=0.01)

In [ ]:
dataloader=DataLoader(dataset=dataset,batch_size=4,shuffle=True,num_workers=2)

# **TRAINING LOOP**

In [ ]:
epochs=100
loss_fn=nn.CrossEntropyLoss()
for epoch in range(epochs):
  for i,(inputs,labels) in enumerate(dataloader):
    labels = labels.squeeze().long()
    outputs=model(inputs)
    optimizer.zero_grad()
    loss=loss_fn(outputs,labels)
    loss.backward()
    optimizer.step()
  if epoch%10==0:
    print(f"epoch{epoch}: {loss.item()}")



epoch0: 5.316344261169434
epoch10: 0.6243249773979187
epoch20: 0.013077490963041782
epoch30: 0.01917901448905468
epoch40: 0.01674852892756462
epoch50: 0.827942967414856
epoch60: 0.01420663669705391
epoch70: 0.01668848656117916
epoch80: 0.1340363621711731
epoch90: 0.005888995714485645


In [ ]:
dataloader=DataLoader(dataset=dataset,batch_size=4,shuffle=True,num_workers=2)
dataiter=iter(dataloader)
data=next(dataiter)
features,labels=data
print(features,data)

tensor([[1.2220e+01, 1.2900e+00, 1.9400e+00, 1.9000e+01, 9.2000e+01, 2.3600e+00,
         2.0400e+00, 3.9000e-01, 2.0800e+00, 2.7000e+00, 8.6000e-01, 3.0200e+00,
         3.1200e+02],
        [1.3670e+01, 1.2500e+00, 1.9200e+00, 1.8000e+01, 9.4000e+01, 2.1000e+00,
         1.7900e+00, 3.2000e-01, 7.3000e-01, 3.8000e+00, 1.2300e+00, 2.4600e+00,
         6.3000e+02],
        [1.3050e+01, 1.7300e+00, 2.0400e+00, 1.2400e+01, 9.2000e+01, 2.7200e+00,
         3.2700e+00, 1.7000e-01, 2.9100e+00, 7.2000e+00, 1.1200e+00, 2.9100e+00,
         1.1500e+03],
        [1.3770e+01, 1.9000e+00, 2.6800e+00, 1.7100e+01, 1.1500e+02, 3.0000e+00,
         2.7900e+00, 3.9000e-01, 1.6800e+00, 6.3000e+00, 1.1300e+00, 2.9300e+00,
         1.3750e+03]]) [tensor([[1.2220e+01, 1.2900e+00, 1.9400e+00, 1.9000e+01, 9.2000e+01, 2.3600e+00,
         2.0400e+00, 3.9000e-01, 2.0800e+00, 2.7000e+00, 8.6000e-01, 3.0200e+00,
         3.1200e+02],
        [1.3670e+01, 1.2500e+00, 1.9200e+00, 1.8000e+01, 9.4000e+01, 2.1000e+0

In [ ]:
features.shape

torch.Size([4, 13])

In [ ]:
labels.shape

torch.Size([4])

In [ ]:
labels=labels.view(1,-1)

In [ ]:
labels.shape

torch.Size([1, 4])

# **MODEL PREDICTION AND EVALUATION**

In [ ]:
dataiter = iter(dataloader)
inputs, labels = next(dataiter)

outputs = model(inputs)
_, predicted = torch.max(outputs, 1)

In [ ]:
print("Predicted:", predicted)
print("Actual   :", labels)

Predicted: tensor([1, 2, 2, 2])
Actual   : tensor([1, 2, 2, 2])


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in dataloader:
        labels = labels.squeeze().long()
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 91.01%
